# grahspj: Fairall 9 With a Fake Photo-z Prior

This notebook uses the real Fairall 9 broadband photometry from the VizieR-based tutorial, but replaces the fixed spectroscopic redshift with a fake bimodal photo-z prior.

Workflow:
- load the Fairall 9 photometric points used in the original tutorial
- build a fake two-Gaussian photo-z PDF with a primary peak near `z ~ 1.5` and a secondary peak near the true redshift
- let `grahspj` fit for redshift using the real photometry
- inspect the inferred redshift and SED fit


## Prerequisites

This notebook assumes:
- you are running from the `notebooks/` directory of this repository
- `grahspj` dependencies are installed
- a valid DSPS SSP file is available

Unlike `02_vizier_fairall9.ipynb`, this notebook does not query VizieR at runtime. The supported Fairall 9 photometry is embedded directly so the only moving piece is the fake photo-z prior.


In [ ]:
import copy
from pathlib import Path
import time
import sys

import matplotlib.pyplot as plt
import numpy as np
from astropy.table import Table

project_root = Path.cwd().resolve().parent
src_root = project_root / "src"
if str(src_root) not in sys.path:
    sys.path.insert(0, str(src_root))

from grahspj.config import AGNConfig, FilterSet, FitConfig, GalaxyConfig, InferenceConfig, LikelihoodConfig, Observation, PhotometryData
from grahspj.core import GRAHSPJ
from grahspj.mplstyle import style_path

plt.style.use(style_path())


In [ ]:
# Real Fairall 9 photometry copied from the VizieR tutorial output.
target_name = "Fairall 9"
true_redshift = 0.04587

phot_rows = [{'vizier_filter': 'WISE:W4', 'grahsp_filter': 'W4', 'speclite_name': 'wise2010-W4', 'psf_fwhm_arcsec': 11.99, 'freq_ghz': 13571.0, 'flux_mjy': 463.99998664855957, 'err_mjy': 7.000000216066837, 'frac_err': 0.015086207796313453, 'catalog': 'II/311/wise'}, {'vizier_filter': 'WISE:W3', 'grahsp_filter': 'W3', 'speclite_name': 'wise2010-W3', 'psf_fwhm_arcsec': 7.36, 'freq_ghz': 25934.0, 'flux_mjy': 241.99999868869781, 'err_mjy': 4.000000189989805, 'frac_err': 0.016528926494480258, 'catalog': 'I/353/gsc242'}, {'vizier_filter': 'WISE:W2', 'grahsp_filter': 'W2', 'speclite_name': 'wise2010-W2', 'psf_fwhm_arcsec': 6.84, 'freq_ghz': 65172.0, 'flux_mjy': 104.99999672174454, 'err_mjy': 1.0000000474974513, 'frac_err': 0.00952381027351366, 'catalog': 'II/365/catwise'}, {'vizier_filter': 'WISE:W1', 'grahsp_filter': 'W1', 'speclite_name': 'wise2010-W1', 'psf_fwhm_arcsec': 6.08, 'freq_ghz': 89490.0, 'flux_mjy': 73.39999824762344, 'err_mjy': 0.7999999797903001, 'frac_err': 0.010899182546182181, 'catalog': 'II/365/catwise'}, {'vizier_filter': '2MASS:Ks', 'grahsp_filter': 'Ks_2mass', 'speclite_name': 'twomass-Ks', 'psf_fwhm_arcsec': 2.5, 'freq_ghz': 138550.0, 'flux_mjy': 45.80000042915344, 'err_mjy': 1.500000013038516, 'frac_err': 0.03275109168085747, 'catalog': 'I/353/gsc242'}, {'vizier_filter': '2MASS:H', 'grahsp_filter': 'H_2mass', 'speclite_name': 'twomass-H', 'psf_fwhm_arcsec': 2.5, 'freq_ghz': 181750.0, 'flux_mjy': 40.19999876618385, 'err_mjy': 1.0999999940395355, 'frac_err': 0.027363184771161064, 'catalog': 'I/353/gsc242'}, {'vizier_filter': '2MASS:J', 'grahsp_filter': 'J_2mass', 'speclite_name': 'twomass-J', 'psf_fwhm_arcsec': 2.5, 'freq_ghz': 241960.0, 'flux_mjy': 25.200000032782555, 'err_mjy': 0.6000000284984708, 'frac_err': 0.023809524909441816, 'catalog': 'VII/275/glade1'}, {'vizier_filter': 'SDSS:i', 'grahsp_filter': 'i_sdss', 'speclite_name': 'sdss2010-i', 'psf_fwhm_arcsec': 1.26, 'freq_ghz': 392660.0, 'flux_mjy': 17.500000074505806, 'err_mjy': 1.7999999690800905, 'frac_err': 0.10285714065237922, 'catalog': 'I/353/gsc242'}, {'vizier_filter': 'SDSS:r', 'grahsp_filter': 'r_sdss', 'speclite_name': 'sdss2010-r', 'psf_fwhm_arcsec': 1.32, 'freq_ghz': 479900.0, 'flux_mjy': 14.100000262260437, 'err_mjy': 2.400000113993883, 'frac_err': 0.17021277087615655, 'catalog': 'I/353/gsc242'}, {'vizier_filter': 'Johnson:V', 'grahsp_filter': 'V_johnson', 'speclite_name': 'bessell-V', 'psf_fwhm_arcsec': None, 'freq_ghz': 541430.0, 'flux_mjy': 12.09999993443489, 'err_mjy': 0.19999999494757503, 'frac_err': 0.01652892529184263, 'catalog': 'J/AJ/138/845/table2'}, {'vizier_filter': 'SDSS:g', 'grahsp_filter': 'g_sdss', 'speclite_name': 'sdss2010-g', 'psf_fwhm_arcsec': 1.44, 'freq_ghz': 621980.0, 'flux_mjy': 10.80000028014183, 'err_mjy': 2.6000000070780516, 'frac_err': 0.2407407351515279, 'catalog': 'I/353/gsc242'}, {'vizier_filter': 'Johnson:B', 'grahsp_filter': 'B_johnson', 'speclite_name': 'bessell-B', 'psf_fwhm_arcsec': None, 'freq_ghz': 674900.0, 'flux_mjy': 12.69999984651804, 'err_mjy': 0.09999999747378752, 'frac_err': 0.007874015644276132, 'catalog': 'J/AJ/138/845/table2'}, {'vizier_filter': 'GALEX:NUV', 'grahsp_filter': 'NUV_galex', 'speclite_name': 'galex-nuv', 'psf_fwhm_arcsec': 5.3, 'freq_ghz': 1296700.0, 'flux_mjy': 5.270000081509352, 'err_mjy': 0.029999999242136255, 'frac_err': 0.005692599388640639, 'catalog': 'I/353/gsc242'}, {'vizier_filter': 'GALEX:FUV', 'grahsp_filter': 'FUV_galex', 'speclite_name': 'galex-fuv', 'psf_fwhm_arcsec': 4.3, 'freq_ghz': 1960700.0, 'flux_mjy': 4.360000137239695, 'err_mjy': 0.03999999898951501, 'frac_err': 0.009174311406063144, 'catalog': 'I/353/gsc242'}]

phot_table = Table(rows=phot_rows)
phot_table


In [ ]:
# Fake bimodal photo-z prior: primary peak at high z, secondary near the truth.
z_grid = np.linspace(0.0, 2.2, 800)
primary_peak = 2.0 * np.exp(-0.5 * ((z_grid - 0.50) / 0.5) ** 2)
secondary_peak = 0.65 * np.exp(-0.5 * ((z_grid - true_redshift) / 0.03) ** 2)
pdf = primary_peak + secondary_peak
pdf /= np.trapezoid(pdf, z_grid)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(z_grid, pdf, lw=2)
ax.axvline(true_redshift, color="k", ls="--", label=f"true z = {true_redshift:.5f}")
ax.axvline(0.50, color="tab:red", ls=":", label="primary fake photo-z peak")
ax.set(xlabel="redshift", ylabel="p(z)", title="Injected fake photo-z prior")
ax.legend()
plt.show()


In [ ]:
dsps_ssp_fn = project_root.parent / "jaxqsofit" / "tempdata.h5"
assert dsps_ssp_fn.is_file(), f"DSPS SSP file not found: {dsps_ssp_fn}"

cfg = FitConfig(
    observation=Observation(
        object_id=f"{target_name} fake photo-z",
        redshift=true_redshift,
        fit_redshift=True,
        redshift_err=0.5,
    ),
    photometry=PhotometryData(
        filter_names=[row["grahsp_filter"] for row in phot_rows],
        fluxes=[row["flux_mjy"] for row in phot_rows],
        errors=[row["err_mjy"] for row in phot_rows],
        is_upper_limit=[False] * len(phot_rows),
        psf_fwhm_arcsec=[row["psf_fwhm_arcsec"] for row in phot_rows],
    ),
    filters=FilterSet(
        speclite_names={row["grahsp_filter"]: row["speclite_name"] for row in phot_rows},
        use_grahsp_database=False,
    ),
    galaxy=GalaxyConfig(dsps_ssp_fn=str(dsps_ssp_fn)),
    agn=AGNConfig(agn_type=1),
    likelihood=LikelihoodConfig(use_host_capture_model=True),
    inference=InferenceConfig(
        map_steps=400,
        learning_rate=5e-3,
        num_warmup=150,
        num_samples=150,
        num_chains=1,
        seed=0,
    ),
    prior_config={
        "log_stellar_mass": {"loc": 10.5, "scale": 1.0},
        "fracAGN_5100": {"loc": 0.8, "scale": 0.15},
        "ebv_gal": {"scale": 0.15},
        "ebv_agn": {"scale": 0.15},
        "redshift_pdf": {"z_grid": z_grid, "pdf": pdf},
    },
)

photoz_fast_cfg = copy.deepcopy(cfg)
photoz_fast_cfg.galaxy.n_wave = 384
photoz_fast_cfg.galaxy.sfh_n_steps = 24
photoz_fast_cfg.galaxy.use_energy_balance = False

print("Configured filters:", cfg.photometry.filter_names)
print("Fit redshift:", cfg.observation.fit_redshift)
print("Initial/reference redshift:", cfg.observation.redshift)
print(
    "Photo-z fast galaxy settings:",
    {
        "n_wave": photoz_fast_cfg.galaxy.n_wave,
        "sfh_n_steps": photoz_fast_cfg.galaxy.sfh_n_steps,
        "use_energy_balance": photoz_fast_cfg.galaxy.use_energy_balance,
    },
)


## Fast Photo-z Configuration Check

### Adaptive grid (profile likelihood) search

In [ ]:
import copy
import numpy as np
import matplotlib.pyplot as plt

def run_profile_point(z, base_cfg, optax_steps):
    cfg_scan = copy.deepcopy(base_cfg)
    cfg_scan.observation.redshift = float(z)
    cfg_scan.observation.fit_redshift = False

    fitter_scan = GRAHSPJ(cfg_scan)
    fitter_scan.fit(
        fit_method="optax",
        prior_config=cfg_scan.prior_config,
        dsps_ssp_fn=cfg_scan.galaxy.dsps_ssp_fn,
        optax_steps=optax_steps,
        optax_lr=cfg_scan.inference.learning_rate,
        plot_fig=False,
        save_fig=False,
        save_result=False,
        progress_bar=False,
        # Keep repeated profile scans cheap; final MAP/NUTS runs use staged MAP by default.
        staged_map=False,
    )
    return float(fitter_scan.map_result["losses"][-1])

# Stage 1: sparse coarse scan
z_coarse = np.linspace(0.0, 2.0, 8)
loss_coarse = np.array([run_profile_point(z, photoz_fast_cfg, optax_steps=40) for z in z_coarse])

print("Coarse scan points and losses:")

# Pick top intervals around the best coarse points
best_idx = np.argsort(loss_coarse)[:3]
refine_points = []

for idx in best_idx:
    z0 = z_coarse[max(idx - 1, 0)]
    z1 = z_coarse[min(idx + 1, len(z_coarse) - 1)]
    refine_points.extend(np.linspace(z0, z1, 7))

z_refine = np.unique(np.concatenate([z_coarse, np.array(refine_points)]))
loss_refine = np.array([run_profile_point(z, photoz_fast_cfg, optax_steps=120) for z in z_refine])

rel_prob = np.exp(-(loss_refine - np.nanmin(loss_refine)))
prior_on_scan = np.interp(z_refine, z_grid, pdf, left=0.0, right=0.0)
pz_profile = rel_prob * prior_on_scan
pz_profile /= np.trapezoid(pz_profile, z_refine)

plt.figure(figsize=(7, 4))
plt.plot(z_refine, pz_profile, marker="o", lw=2, label="adaptive profile p(z)")
plt.plot(z_grid, pdf / np.trapezoid(pdf, z_grid), lw=2, alpha=0.7, label="injected prior")
plt.axvline(true_redshift, color="k", ls="--", label=f"true z = {true_redshift:.5f}")
plt.xlabel("redshift")
plt.ylabel("density")
plt.legend()
plt.show()


### Nested Sampling method

In [ ]:
fitter = GRAHSPJ(cfg)
fit_result = fitter.fit(
    fit_method="ns",
    prior_config=cfg.prior_config,
    dsps_ssp_fn=cfg.galaxy.dsps_ssp_fn,
    optax_steps=cfg.inference.map_steps,
    optax_lr=cfg.inference.learning_rate,
    nuts_warmup=cfg.inference.num_warmup,
    nuts_samples=cfg.inference.num_samples,
    nuts_chains=cfg.inference.num_chains,
    ns_live_points=200,
    ns_max_samples=3000,
    ns_dlogz=10,
    plot_fig=False,
    save_fig=False,
    save_result=False,
    progress_bar=True,
)


In [ ]:
samples = fit_result["fit"]["results"].samples

if "redshift_fit" not in samples:
    raise KeyError(f"redshift_fit not found. Available sample keys include: {list(samples)[:20]}")

z_samples = np.asarray(samples["redshift_fit"], dtype=float)

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(
    z_samples,
    bins=40,
    density=True,
    alpha=0.7,
    color="tab:blue",
    label="posterior samples",
)
ax.axvline(true_redshift, color="k", ls="--", label=f"true z = {true_redshift:.5f}")
ax.set(
    xlabel="redshift",
    ylabel="density",
    title=f"{target_name}: redshift posterior",
)
ax.legend()
plt.show()


## Notes

- The photometry here is real Fairall 9 photometry from the original tutorial; only the redshift prior is fake.
- If `optax+nuts` is too slow, use `fit_method="optax"` first. MAP/SVI initialization is staged by default; pass `staged_map=False` only for fast profile scans or old-path comparisons.
- To make the prior more or less misleading, adjust the amplitudes, widths, and centers in the `primary_peak` and `secondary_peak` definitions.
